# YOLO26n 원판 탐지 베이스라인

검증 프레임 한 장의 정답과 사전학습 모델의 예측을 먼저 비교합니다. 이후 같은 장면에서 원판 학습 후의 변화를 살펴봅니다.

## 0. 영상 입력 검사

파일 이름이 `.mp4`로 끝나도 내부 컨테이너와 코덱은 다를 수 있습니다. 먼저 파일에서 읽은 헤더 브랜드와 코덱을 확인하면, 확장자가 잘못 붙었거나 영상 스트림을 읽을 수 없는 파일을 학습 전에 찾을 수 있습니다.

전송이 중간에 끊긴 영상은 헤더가 읽혀도 뒷부분이 누락될 수 있습니다. 그래서 학습 코드와 같은 방식으로 첫 프레임부터 **마지막 라벨 프레임**까지 읽습니다. 번호는 0부터 셉니다. `프레임 읽기`가 실패하거나 미검사이면 `메모`의 원인을 먼저 확인하세요. `QuickTime 브랜드(qt)`는 파일 형식에 관한 안내입니다.

이 셀에는 FFmpeg의 `ffprobe`가 필요합니다. Mac에서 설치되어 있지 않으면 터미널에서 `brew install ffmpeg`를 실행하세요.

In [ ]:
from wlcv.config import ANNOTATIONS, VIDEOS
from wlcv.data.video_inputs import inspect_video_inputs

video_checks = inspect_video_inputs(VIDEOS, ANNOTATIONS / "plate_boxes")
display(video_checks)

In [ ]:
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, DATASET, RUNS, SPLIT, VIDEOS, WEIGHTS
from wlcv.data.dataset import prepare_dataset
from wlcv.data.plate_labels import label_frame
from wlcv.data.frames import read_rgb
from wlcv.evaluation.plots import plot_comparison, plot_training, predict_rgb
from wlcv.training.train import train_baseline

## 1. 정답과 사전학습 예측 비교

초록색 테두리가 사람이 표시한 원판입니다. 오른쪽에는 COCO로 사전학습한 모델이 인식한 물체와 신뢰도가 표시됩니다. 다른 장면을 보려면 라벨 파일명을 바꾸세요.

In [ ]:
labels = ANNOTATIONS / "plate_boxes"
label = labels / "a03_sn_f_01_f2.txt"
video_id, frame_index = label_frame(label)
frame = read_rgb(VIDEOS / f"{video_id}.mp4", frame_index)
pretrained = YOLO(str(WEIGHTS.resolve(strict=True)))
before = predict_rgb(pretrained, frame)
plot_comparison(frame, label, before, "YOLO26n (COCO)")

## 2. 데이터 준비와 학습 실행 선택

위에서 원판 위치를 확인한 뒤 이 셀을 실행하세요. 사람별 분할표에 따라 라벨이 있는 프레임을 준비합니다. 학습은 `run_training = True`로 바꾸고 이 셀을 다시 실행할 때 시작합니다. 기존 실행을 검수할 때는 `run_dir`에 그 폴더를 지정하세요.

In [ ]:
run_training = False
run_dir = RUNS / "plate_baseline"
data_yaml = prepare_dataset(VIDEOS, labels, SPLIT, DATASET)

if run_training:
    run_dir = train_baseline(data_yaml, name=run_dir.name)
print("실험 폴더", run_dir)

## 3. 학습 기록

왼쪽은 학습셋과 검증셋의 손실 합계입니다. 오른쪽은 검증셋의 mAP, 정밀도, 재현율입니다.

In [ ]:
if (run_dir / "results.csv").is_file():
    plot_training(run_dir)
else:
    print("학습 기록이 없습니다. 위 셀에서 학습하거나 기존 실험 폴더를 지정하세요.")

## 4. 같은 장면의 학습 후 예측

검증 성능으로 선택한 `best.pt`를 불러옵니다. 첫 비교와 동일한 프레임, 입력 크기, 신뢰도 기준을 사용합니다.

In [ ]:
best = run_dir / "weights" / "best.pt"
detector = None
if best.is_file():
    detector = YOLO(str(best))
    after = predict_rgb(detector, frame)
    plot_comparison(frame, label, after, "YOLO26n (plate)")
else:
    print("이 실험 폴더에 best.pt가 없습니다.")

## 5. FiftyOne에서 검증 프레임 둘러보기

`open_review = True`로 바꾸면 로컬 검수 화면이 열립니다. 학습 전에는 `ground_truth`로 정답 박스를 봅니다. 학습 후에는 위 셀에서 불러온 모델의 `predictions`도 함께 표시됩니다.

`person_id`로 사람을 선택하고 `video_id`, `frame_index`로 원본 장면을 찾으세요. 이 저신뢰 검수 예시에서는 예측 신뢰도 0.10 이상에서 정답과 예측의 겹침(IoU)이 0.5 이상이면 일치로 셉니다. `eval_fn`이 1 이상이면 놓친 원판, `eval_fp`가 1 이상이면 정답과 대응되지 않은 예측이 있는 프레임입니다.

In [ ]:
open_review = False

if open_review:
    import fiftyone as fo

    from wlcv.evaluation.fiftyone_review import evaluate_plates, load_validation_frames

    review = load_validation_frames(data_yaml, SPLIT)
    if detector is not None:
        evaluate_plates(review, detector, confidence=0.10)
        print("오탐 박스", review.sum("eval_fp"), "미탐 박스", review.sum("eval_fn"))
    print("검증 프레임", len(review))
    session = fo.launch_app(review, address="127.0.0.1")

In [ ]:
if open_review and detector is not None:
    from fiftyone import ViewField as F

    video_id = "a03_sn_f_01"
    frames = review.match(F("video_id") == video_id).sort_by("frame_index")

    for sample in frames:
        scores = [round(box.confidence, 3) for box in sample.predictions.detections]
        print(f"프레임 {sample.frame_index:>3}  {scores}")